# 01 · Build the Modelling Dataset

Turns the raw loan-level data of the thesis (`Δεδομένα/Updated.xlsx`, sheet `dataset`) into
`outputs/model_dataset.csv`, which notebooks 02–04 read.

1. Load and inspect the raw data
2. Date features
3. Categorical encoding (no dummy-variable trap)
4. Train / validation / test split, assigned **before** any statistic is computed
5. Correlation analysis on the training rows
6. Recovery-rate target and a check against the thesis dataset
7. Which features are known at the moment of default
8. Information Value and AUC on the training rows
9. Save

Features are **not scaled here**. Scaling is fitted on the training rows inside each model, so that the
validation and test rows never influence it.

In [ ]:
# Reload the lgd package automatically when its files change (no kernel restart needed after edits).
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from lgd.config import (MODEL_DATA_FILE, OUTPUT_DIR, POST_DEFAULT_FEATURES, RAW_DATA_FILE, TARGETS,
                        THESIS_DATA_FILE)
from lgd.data import (CORRELATED_FEATURES_TO_DROP, CORRELATION_THRESHOLD, add_date_features, assign_split,
                      encode_categoricals, high_correlation_pairs)
from lgd.plots import save_and_show

pd.set_option("display.max_columns", 100)

## 1. Load and Inspect the Raw Data

In [ ]:
raw = pd.read_excel(RAW_DATA_FILE, sheet_name="dataset")
raw.shape

In [ ]:
raw.info()

In [ ]:
raw.describe().T

In [ ]:
missing = raw.isnull().sum()
missing[missing > 0] if missing.any() else "No missing values"

## 2. Date Features

From the origination (`DATE_OF_LOAN`), default (`DATE_FROM`) and write-off (`DATE_WO`) dates we keep the
month of each event and the number of days between them.

In [ ]:
dataset = add_date_features(raw)
dataset[["loan_month", "default_month", "wo_month", "loan_to_default_days", "loan_to_wo_days", "default_to_wo_days"]].describe().T

## 3. Categorical Encoding

| Feature | Encoding |
|---|---|
| `home_ownership`, `marital_status_code`, `city` | One-hot; the most frequent category is the reference and is dropped |
| `country` | Dropped: it is fully determined by `city` |
| `postal_code` | Dropped: `city` was derived from it |
| `grade`, `sub_grade` | Label codes, only for the correlation analysis (dropped there) |

In the thesis dataset both all country dummies *and* all city dummies were kept. Each group sums to 1, so the
columns were linearly dependent (dummy-variable trap) and the linear-model coefficients were not unique.
`home_ownership` was label-encoded (ANY < MORTGAGE < OWN < RENT), an order that does not exist; it is now
one-hot encoded.

In [ ]:
dataset = encode_categoricals(dataset)
print(dataset.shape)
[col for col in dataset.columns if col.startswith(("home_ownership_", "marital_status_code_", "city_"))]

## 4. Train / Validation / Test Split

70% / 10% / 20%, stratified on `cure`. The split is stored in the `split` column, so all notebooks use
exactly the same loans. From here on, every statistic used to make a modelling decision is computed on
the **training rows only**.

In [ ]:
dataset.insert(0, "split", assign_split(dataset))
train_rows = dataset["split"] == "train"

dataset.groupby("split")["cure"].agg(loans="size", cure_rate="mean")

## 5. Correlation Analysis (training rows)

Pairs with |correlation| above 80% (thesis, Table 5); one feature of each pair is removed.
The one-hot columns are left out of the heatmap so that it stays readable. The `rec_rate` here is the
original one from `Updated.xlsx` (see section 6).

In [ ]:
correlation_matrix = dataset.loc[train_rows].drop(columns="split").corr()

dense_columns = [col for col in correlation_matrix.columns
                 if not col.startswith(("home_ownership_", "marital_status_code_", "city_"))]
fig, ax = plt.subplots(figsize=(19.2, 10.8))
sns.heatmap(correlation_matrix.loc[dense_columns, dense_columns], annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, annot_kws={"size": 6}, ax=ax)
ax.set_title("Correlation Matrix Heatmap (training rows)")
save_and_show(fig, "correlation_matrix.png")

In [ ]:
high_correlation_pairs(correlation_matrix, CORRELATION_THRESHOLD)

In [ ]:
dataset = dataset.drop(columns=CORRELATED_FEATURES_TO_DROP)

## 6. Recovery-Rate Target and Check Against the Thesis Dataset

The models of the thesis were trained on `Dataset_with_rec_rate.xlsx`. Its `rec_rate` column is a newer
recovery rate that is **not** part of `Updated.xlsx`, so it is taken from that file (the rows are in the same
order; the check below confirms that every shared feature matches).

In [ ]:
thesis_dataset = pd.read_excel(THESIS_DATA_FILE)

# The thesis file is Min-Max scaled over all rows (except Euribor_3), so rescale our columns the same way for the check.
shared = [col for col in thesis_dataset.columns if col in dataset.columns and col not in TARGETS]
ours = dataset[shared].astype(float)
ours_scaled = (ours - ours.min()) / (ours.max() - ours.min())
ours_scaled["Euribor_3"] = ours["Euribor_3"]

largest_difference = (ours_scaled - thesis_dataset[shared]).abs().max().max()
print(f"{len(shared)} shared features, largest difference: {largest_difference:.2e}")
assert largest_difference < 1e-6
assert dataset["cure"].equals(thesis_dataset["cure"])

print(f"rec_rate largest difference (Updated.xlsx vs. thesis file): {(dataset['rec_rate'] - thesis_dataset['rec_rate']).abs().max():.3f}")
dataset["rec_rate"] = thesis_dataset["rec_rate"]

## 7. Which Features Are Known at the Moment of Default?

An LGD model is applied when a loan defaults. Features that are only known after the workout has ended
(recoveries, payments received "to date", write-off amount and dates, whether the borrower is healthy again)
leak the outcome into the model. Notebooks 02 and 03 use the **`at_default`** feature set by default and
report how much the **`thesis`** feature set (all features) inflates the results.

In [ ]:
pd.DataFrame({"feature": POST_DEFAULT_FEATURES}).assign(known_at_default=False)

## 8. Information Value and AUC for `cure` (training rows)

* **IV**: features with more than 10 distinct values are binned into deciles first; empty bins are
  smoothed so that the Weight of Evidence never becomes `log(0)`.
* **AUC**: computed directly on the feature values and reported as `max(AUC, 1 − AUC)`, so the direction of
  the relationship does not matter.

Rule of thumb for IV: < 0.02 useless · 0.02–0.1 weak · 0.1–0.3 medium · 0.3–0.5 strong ·
**> 0.5 suspicious** (too good to be true; check for leakage or for how the data was generated).

In [ ]:
def information_value(feature: pd.Series, target: pd.Series, bins: int = 10, smoothing: float = 0.5) -> float:
    if feature.nunique() > bins:
        feature = pd.qcut(feature, q=bins, duplicates="drop")
    counts = pd.crosstab(feature, target) + smoothing
    distribution_good = counts[0] / counts[0].sum()
    distribution_bad = counts[1] / counts[1].sum()
    woe = np.log(distribution_good / distribution_bad)
    return float(((distribution_good - distribution_bad) * woe).sum())


def feature_auc(feature: pd.Series, target: pd.Series) -> float:
    auc = roc_auc_score(target, feature)
    return max(auc, 1 - auc)


def iv_strength(iv: float) -> str:
    for limit, label in [(0.02, "useless"), (0.1, "weak"), (0.3, "medium"), (0.5, "strong")]:
        if iv < limit:
            return label
    return "suspicious"


train = dataset[dataset["split"] == "train"]
feature_columns = [col for col in dataset.columns if col not in TARGETS + ["split"]]
iv_auc = pd.DataFrame(
    [{"Feature": col,
      "IV": information_value(train[col], train["cure"]),
      "AUC": feature_auc(train[col], train["cure"]),
      "Known at default": col not in POST_DEFAULT_FEATURES}
     for col in feature_columns]
).sort_values("IV", ascending=False, ignore_index=True)
iv_auc["Strength"] = iv_auc["IV"].map(iv_strength)

iv_auc.to_csv(OUTPUT_DIR / "feature_iv_auc.csv", index=False)
iv_auc.head(20)

> **LTV** on its own separates cured from non-cured loans almost perfectly (very high IV, AUC ≈ 0.99). LTV is
> known at default, so it stays in the model, but this explains most of the ~98–99% accuracy of the cure
> models and should be discussed in the thesis (real portfolios rarely show one feature this strong).

## 9. Save

In [ ]:
dataset = dataset[["split"] + feature_columns + TARGETS]
dataset.index.name = "loan_id"
dataset.to_csv(MODEL_DATA_FILE)
print(f"Saved {dataset.shape[0]} loans x {len(feature_columns)} features to {MODEL_DATA_FILE}")